# 02 · Axe 2 Confiance et vérification (Antoine REVERSAT)

**Questions :** quelle confiance accorde-t-on à l'IA ? la capacité perçue a-t-elle changé depuis 2025 ? est-ce robuste ?

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))   # le notebook vit dans notebooks/ : on importe devia depuis la racine
import pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.width", 160, "display.max_columns", 30)
from devia import data, population
d26 = data.prepare_2026(data.load_2026()); d25 = data.prepare_2025(data.load_2025())
p26, p25 = d26[d26.is_pro_dev], d25[d25.is_pro_dev]          # population par défaut : développeurs professionnels
print(f"2026 : {len(p26):,} développeurs professionnels sur {len(d26):,} répondants | 2025 : {len(p25):,} sur {len(d25):,}")
from devia import indicators
from devia.axes import confiance

2026 : 20,220 développeurs professionnels sur 30,903 répondants | 2025 : 34,146 sur 49,191


In [2]:
confiance.trust_distribution(p26).round(3)

,category,n,k,share,lo,hi
0,Seulement si je peux vérifier facilement,10150,4931,0.486,0.476,0.496
1,Tâches à faible risque seulement,10150,1566,0.154,0.147,0.161
2,"Beaucoup de tâches, pas les décisions importantes",10150,1666,0.164,0.157,0.171
3,Y compris les décisions importantes,10150,698,0.069,0.064,0.074
4,Pas de confiance pour la plupart des tâches,10150,1289,0.127,0.121,0.134


Près d'un répondant sur deux ne fait confiance à l'IA que s'il peut vérifier le résultat ; très peu l'acceptent pour des décisions importantes.

In [3]:
confiance.compare_capability(p26, p25).round(3)

,category,share_2025,lo_2025,hi_2025,n_2025,share_2026,lo_2026,hi_2026,n_2026
0,Très mauvaise,0.218,0.213,0.224,20397,0.084,0.079,0.090,9598
1,Mauvaise,0.270,0.263,0.276,20397,0.169,0.162,0.177,9598
2,Ni bonne ni mauvaise,0.170,0.164,0.175,20397,0.149,0.142,0.156,9598
3,"Bonne, sans plus",0.298,0.292,0.304,20397,0.420,0.410,0.430,9598
4,Très bonne,0.045,0.042,0.048,20397,0.178,0.170,0.186,9598


## Robustesse : est-ce un effet de composition ?
On refait la comparaison parmi les seuls utilisateurs d'IA (2025 : réponses « Yes… » à `AISelect`).

In [4]:
raw25 = pd.read_csv("../data/2025/results.csv", usecols=["ResponseId","AISelect"], encoding="utf-8-sig")
u25 = p25.merge(raw25, on="ResponseId"); u25 = u25[u25.AISelect.fillna("").str.startswith("Yes")]
u26 = p26[p26.ai_user == 1]
print("Capacité positive, utilisateurs : 2025 =", round(u25.capability_positive.mean(),3), "| 2026 =", round(u26.capability_positive.mean(),3))
print("Attitude favorable, utilisateurs : 2025 =", round((u25.sentiment3=="Favorable").sum()/u25.sentiment3.notna().sum(),3), "| 2026 =", round((u26.sentiment3=="Favorable").sum()/u26.sentiment3.notna().sum(),3))

Capacité positive, utilisateurs : 2025 = 0.366 | 2026 = 0.598
Attitude favorable, utilisateurs : 2025 = 0.74 | 2026 = 0.638


**Lecture :** la hausse de capacité perçue tient parmi les utilisateurs. En revanche, l'attitude favorable parmi les seuls utilisateurs est plus basse en 2026 qu'en 2025, alors qu'elle est stable sur l'ensemble : les définitions d'« utilisateur » diffèrent, nous ne concluons pas (voir `DECISIONS.md` §6).

In [5]:
confiance.positive_capability_by(p26, p25, "exp_band").round(3)

,group,n_2025,k_2025,share_2025,lo_2025,hi_2025,n_2026,k_2026,share_2026,lo_2026,hi_2026
0,0-2 ans,1579,577,0.365,0.342,0.389,652,345,0.529,0.491,0.567
1,3-5 ans,3355,1229,0.366,0.350,0.383,1232,676,0.549,0.521,0.576
2,6-10 ans,4959,1686,0.340,0.327,0.353,2034,1185,0.583,0.561,0.604
3,11-20 ans,6224,2033,0.327,0.315,0.338,3206,1931,0.602,0.585,0.619
4,21 ans et +,4011,1345,0.335,0.321,0.350,2406,1561,0.649,0.629,0.668
